# 03 — Gold Exploration

**Run this AFTER dbt Cloud has executed all models.**

Validates the Gold star schema and answers product marketing questions:
- Genre performance
- Publisher market share
- Regional sales split
- Annual sales trend
- Platform rankings

In [ ]:
# Confirm Gold tables exist
spark.sql('SHOW TABLES IN gaming_gold').show(truncate=False)

In [ ]:
# --- Q1: Top genres by global sales ---
display(spark.sql("""
    SELECT g.genre_name,
           ROUND(SUM(f.global_sales_millions),2) AS total_sales_M,
           COUNT(*) AS game_count,
           g.genre_category,
           g.is_competitive
    FROM gaming_gold.fct_game_sales f
    JOIN gaming_gold.dim_genre g ON f.genre_id = g.genre_id
    GROUP BY g.genre_name, g.genre_category, g.is_competitive
    ORDER BY total_sales_M DESC
"""))

In [ ]:
# --- Q2: Publisher market share (Top 15) ---
display(spark.sql("""
    SELECT p.publisher_name,
           ROUND(SUM(f.global_sales_millions),2) AS total_sales_M,
           COUNT(*) AS title_count,
           ROUND(SUM(f.global_sales_millions) /
                 SUM(SUM(f.global_sales_millions)) OVER() * 100, 2) AS market_share_pct
    FROM gaming_gold.fct_game_sales f
    JOIN gaming_gold.dim_publisher p ON f.publisher_id = p.publisher_id
    GROUP BY p.publisher_name
    ORDER BY total_sales_M DESC
    LIMIT 15
"""))

In [ ]:
# --- Q3: Regional sales split by genre ---
display(spark.sql("""
    SELECT g.genre_name,
           ROUND(SUM(f.na_sales_millions),2)    AS NA_M,
           ROUND(SUM(f.eu_sales_millions),2)    AS EU_M,
           ROUND(SUM(f.jp_sales_millions),2)    AS JP_M,
           ROUND(SUM(f.other_sales_millions),2) AS Other_M,
           ROUND(SUM(f.global_sales_millions),2) AS Global_M
    FROM gaming_gold.fct_game_sales f
    JOIN gaming_gold.dim_genre g ON f.genre_id = g.genre_id
    GROUP BY g.genre_name
    ORDER BY Global_M DESC
"""))

In [ ]:
# --- Q4: Annual sales trend ---
display(spark.sql("""
    SELECT g.year_of_release,
           ROUND(SUM(f.global_sales_millions),2) AS total_sales_M,
           COUNT(*) AS titles_released
    FROM gaming_gold.fct_game_sales f
    JOIN gaming_gold.dim_game g ON f.game_id = g.game_id
    WHERE g.year_of_release BETWEEN 1990 AND 2016
    GROUP BY g.year_of_release
    ORDER BY g.year_of_release
"""))

In [ ]:
# --- Q5: Platform rankings ---
display(spark.sql("""
    SELECT p.platform_name,
           ROUND(SUM(f.global_sales_millions),2) AS total_sales_M,
           COUNT(*) AS title_count
    FROM gaming_gold.fct_game_sales f
    JOIN gaming_gold.dim_platform p ON f.platform_id = p.platform_id
    GROUP BY p.platform_name
    ORDER BY total_sales_M DESC
    LIMIT 20
"""))

In [ ]:
# --- Q6: Blockbuster vs Long-Tail split ---
display(spark.sql("""
    SELECT sales_tier,
           COUNT(*) AS game_count,
           ROUND(SUM(global_sales_millions),2) AS total_sales_M,
           ROUND(AVG(global_sales_millions),3) AS avg_sales_M
    FROM gaming_gold.fct_game_sales
    GROUP BY sales_tier
    ORDER BY total_sales_M DESC
"""))